In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

In [ ]:
hourly_subregion_demand_list = []
for year in range(2016, 2024):
    df = pd.read_csv(
        f"../data/baseline_load_profiles/raw/{year}_hourly_demand_by_subregion.csv",
        dtype={'subba': str},
        parse_dates=['timestamp']
    )
    hourly_subregion_demand_list.append(df)

hourly_subregion_demand = (
    pd.concat(hourly_subregion_demand_list)
    .reset_index(drop=True)
    .rename(columns={'timestamp': 'date_time'})
)
all_subregions = list(
    hourly_subregion_demand.loc[hourly_subregion_demand.parent != 'PNM']
    .subba
    .unique()
)

In [ ]:
all_regions = [
    'AEC', 'AECI', 'CPLE', 'CPLW',
    'DUK', 'FMPP', 'FPC',
    'FPL', 'GVL', 'HST', 'ISNE',
    'JEA', 'LGEE', 'MISO', 'NSB',
    'NYIS', 'PJM', 'SC',
    'SCEG', 'SOCO',
    'SPA', 'SWPP', 'TAL', 'TEC',
    'TVA', 'ERCO',
    'AVA', 'AZPS', 'BANC', 'BPAT',
    'CHPD', 'CISO', 'DOPD',
    'EPE', 'GCPD', 'IID',
    'IPCO', 'LDWP', 'NEVP', 'NWMT',
    'PACE', 'PACW', 'PGE', 'PNM',
    'PSCO', 'PSEI', 'SCL', 'SRP',
    'TEPC', 'TIDC', 'TPWR', 'WACM',
    'WALC', 'WAUW'
]

In [ ]:
# Assume the MICE results file is a subset of the original hours
def trim_rows_to_match_length(mice, df):
    mice_start = mice.loc[0, 'date_time']
    mice_end = mice.loc[len(mice.index)-1, 'date_time']
    to_drop = []
    for idx in df.index:
        if df.loc[idx, 'date_time'] != mice_start:
            to_drop.append(idx)
        else: # stop once equal
            break
    for idx in reversed(df.index):
        if df.loc[idx, 'date_time'] != mice_end:
            to_drop.append(idx)
        else: # stop once equal
            break
    
    df = df.drop(to_drop, axis=0)
    df = df.reset_index()
    assert(len(mice.index) == len(df.index))
    return df

def distribute_MICE_results(profile_type):
    raw_demand_file_loc=f'data/{profile_type}/raw_inputs'
    screening_file=f'data/{profile_type}/csv_MASTER.csv'
    mice_results_csv=f'data/{profile_type}/MICE_output/mean_impute_csv_MASTER.csv'
    out_dir=f'data/{profile_type}/outputs'

    os.makedirs(out_dir, exist_ok=True)
    
    # Load screening results
    screening = pd.read_csv(screening_file)
    screening['date_time'] = pd.to_datetime(screening['date_time']).astype(str)
    # Load MICE results
    mice = pd.read_csv(mice_results_csv)
    mice['date_time'] = pd.to_datetime(mice['date_time'], format='ISO8601').astype(str)

    screening = trim_rows_to_match_length(mice, screening)
    
    # Distribute to single BA results files first
    print("Distribute MICE results per-respondent:")
    for fname in os.listdir(raw_demand_file_loc):
        respondent = fname.split('.')[0]

        if respondent in ['OVEC', 'SEC']:
            continue
        
        print(respondent)

        if respondent.isnumeric():
            mice_demand_col = 'X' + str(int(respondent)).zfill(4)
        else:
            mice_demand_col = respondent

        
        df = pd.read_csv(os.path.join(raw_demand_file_loc, fname))      
        df = df.drop_duplicates()
        try:
            df = trim_rows_to_match_length(mice, df)
        except AssertionError:
            print(f"Skipping {respondent}.")
    
        df_out = pd.DataFrame({
            'date_time': df['date_time'],
            'raw demand (MW)': df['demand (MW)'],
            'category': screening[f'{respondent}_category'],
            'cleaned demand (MW)': mice[mice_demand_col]
        })
        
        df_out.to_csv(f'{out_dir}/{respondent}.csv', index=False)

In [ ]:
# Options: [regional, subregional, forecast, load_loss_correction]
profile_type = ''
assert profile_type != '', "Must set a profile_type (options: regional, subregional, forecast, or load_loss_correction)"

distribute_MICE_results(profile_type)